In [ ]:
from apscheduler.schedulers.blocking import BlockingScheduler
from zoneinfo import ZoneInfo
from datetime import datetime, timedelta
import math
from holidays_utils import market_status, market_holidays
from get_symbols import get_active_symbols, update_symbols, get_top_symbols_by_market_cap
from intraday import update_intraday_for_symbols, update_intraday_for_symbols_batched
from daily import update_daily_data
from archive_job import archive_old_intraday
from time_utils import now_ny, in_work_window, delayed_timestamp
import time as _time

NY = ZoneInfo("America/New_York")
sched = BlockingScheduler(timezone=NY)

# 你想要的工作時段（NY 時區）
WORK_START = (9, 45)  # 09:45
WORK_END = (16, 15)  # 16:15

TOP_MARKET_CAP_COUNT = 20
INTRADAY_DELAY_MINUTES = 15
BATCH_SIZE = 25
BATCH_INTERVAL_SECONDS = 30
SLOW_INTERVALS = ["15m", "30m", "1h", "4h"]


def get_batch_index(total_symbols: int, batch_size: int, batch_interval_seconds: int, now=None) -> int:
    if total_symbols <= 0:
        return 0
    batches = max(1, math.ceil(total_symbols / batch_size))
    if now is None:
        now = now_ny()
    tick = int(now.timestamp() // batch_interval_seconds)
    return tick % batches


# 每分鐘 1m job（job 內會先檢查 market_status 與工作時段）
@sched.scheduled_job("cron", minute="*")
def job_intraday_1m():
    try:
        status = market_status()
    except Exception as e:
        print("market_status error:", e)
        return
    if not status.get("marketOpen", False):
        return
    if not in_work_window(start_hm=WORK_START, end_hm=WORK_END):
        return
    symbols = get_top_symbols_by_market_cap(TOP_MARKET_CAP_COUNT)
    fetch_until = delayed_timestamp(INTRADAY_DELAY_MINUTES)
    update_intraday_for_symbols(symbols, "1m", fetch_until=fetch_until)


# 每 5 分鐘 5m job
@sched.scheduled_job("cron", minute="*/5")
def job_intraday_5m():
    try:
        status = market_status()
    except Exception as e:
        print("market_status error:", e)
        return
    if not status.get("marketOpen", False):
        return
    if not in_work_window(start_hm=WORK_START, end_hm=WORK_END):
        return
    symbols = get_top_symbols_by_market_cap(TOP_MARKET_CAP_COUNT)
    fetch_until = delayed_timestamp(INTRADAY_DELAY_MINUTES)
    update_intraday_for_symbols(symbols, "5m", fetch_until=fetch_until)


# 分批更新 15m/30m/1h/4h（每 30 秒跑一批）
@sched.scheduled_job("interval", seconds=BATCH_INTERVAL_SECONDS)
def job_intraday_slow_batch():
    try:
        status = market_status()
    except Exception as e:
        print("market_status error:", e)
        return
    if not status.get("marketOpen", False):
        return
    if not in_work_window(start_hm=WORK_START, end_hm=WORK_END):
        return

    symbols = get_active_symbols()
    if not symbols:
        return
    now = now_ny()
    interval_index = int(now.timestamp() // BATCH_INTERVAL_SECONDS) % len(SLOW_INTERVALS)
    interval = SLOW_INTERVALS[interval_index]
    batch_index = get_batch_index(len(symbols), BATCH_SIZE, BATCH_INTERVAL_SECONDS, now=now)
    fetch_until = delayed_timestamp(INTRADAY_DELAY_MINUTES, now=now)
    update_intraday_for_symbols_batched(
        symbols,
        interval,
        batch_size=BATCH_SIZE,
        batch_index=batch_index,
        fetch_until=fetch_until,
    )


# daily update：market close + 1hr buffer（17:05 NY time）
@sched.scheduled_job("cron", hour=17, minute=5)
def job_daily_update():
    try:
        status = market_status()
    except Exception as e:
        print("market_status error:", e)
        return
    if status.get("marketOpen", True):
        return
    update_daily_data()


# archive job：每日凌晨 NY time 02:00
@sched.scheduled_job("cron", hour=2, minute=0)
def job_archive():
    archive_old_intraday()


# symbols update：每日凌晨 NY time 02:10
@sched.scheduled_job("cron", hour=2, minute=10)
def job_update_symbols():
    update_symbols()


if __name__ == "__main__":
    print("Scheduler starting (NY time)...")
    sched.start()